<img src="images/header.png" width=180, align="center"/>

Master's degree in Intelligent Systems

Subject: 11754 - Deep Learning

Year: 2025-2026

Professor: Miguel Ángel Calafat Torrens

# LESSON 3B — WGAN-GP and GAN Evaluation

In LESSON 3A, we saw that classical GANs suffer from training instability and mode collapse. In this notebook, we address these problems with the **Wasserstein GAN with Gradient Penalty (WGAN-GP)** and compare both approaches side-by-side.

In [ ]:
# Environment detection: Colab vs local
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running on {"Google Colab" if IN_COLAB else "local environment"}')

In [ ]:
# Setup: Drive mount (Colab) or local path
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')
    %cd '/content/gdrive/MyDrive/LABS2026/LAB03'

import pathlib
import sys
import os

PROJECT_DIR = str(pathlib.Path().resolve())
sys.path.append(PROJECT_DIR)

import helper_L3 as hp

In [ ]:
# Imports
import PIL
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
# Reload helper in case it was modified during development
import importlib
importlib.reload(hp)

SEED = 42
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# 1. CelebA Dataset Setup

The classes `CustomDataset` and `ToScaledTensor` are defined in `helper_L3.py` and accessed via the `hp` alias (e.g., `hp.CustomDataset(...)`), so there is no need to redefine them here.

In [ ]:
# Dataset zip path
if IN_COLAB:
    dataset_zip_fullpath = '/content/gdrive/MyDrive/datasets/img_align_celeba_small_10000.zip'
else:
    dataset_zip_fullpath = os.path.join(PROJECT_DIR, '..', 'datasets', 'img_align_celeba_small_10000.zip')

DATA_DIR = hp.extract_dataset(dataset_zip_fullpath, remove_zip=IN_COLAB)


In [ ]:
# Transforms and dataset — using helper classes
img_size = 64
transform = transforms.Compose([
    transforms.CenterCrop((178, 178)),
    transforms.Resize((img_size, img_size)),
    hp.ToScaledTensor(),
])

dataset = hp.CustomDataset(DATA_DIR, transforms=transform, lim=-1)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

In [ ]:
hp.show(next(iter(dataloader))[0])

# 2. Why Wasserstein? The Problem with BCE Loss

## When the Discriminator Wins Too Easily

In a classical GAN, the discriminator outputs a probability via a sigmoid function. When the discriminator becomes too good at distinguishing real from fake, the gradients that flow back to the generator **vanish**. This happens because:

- The JS divergence (which BCE implicitly optimizes) **saturates** when the real and fake distributions don't overlap.
- The gradient signal becomes essentially zero, and the generator receives no useful information to improve.

This is the fundamental instability of classical GANs: the better the discriminator gets, the worse the gradient signal for the generator.

## Earth Mover's Distance (Wasserstein-1)

The **Wasserstein distance** (also called the Earth Mover's Distance) measures the minimum "cost" to transport one probability distribution to another. Imagine two piles of earth: the Wasserstein distance is the minimum work needed to reshape one pile into the other.

<img src="images/wasserstein_distance.png" width=700, align="center"/>

Unlike the KL or JS divergence, the Wasserstein distance:
- Provides **smooth, meaningful gradients** even when the distributions don't overlap.
- Gives a **continuous metric** that decreases as distributions get closer, even before they overlap.

This makes it a much better training signal for the generator.

## The 1-Lipschitz Constraint

To compute the Wasserstein distance, the critic function must be **1-Lipschitz continuous**
— meaning its output can't change faster than the input changes. Two approaches to enforce this:

1. **Weight clipping** (original WGAN): Crude — forces all weights into a small range
   (e.g., [-0.01, 0.01]). This under-utilizes network capacity, can cause vanishing or
   exploding gradients, and pushes weights to the boundary values.
2. **Gradient penalty** (WGAN-GP): Penalizes the critic when its gradients deviate from
   unit norm on interpolated samples. This is more flexible and produces better results.

We use the gradient penalty approach (WGAN-GP).

## Wasserstein Loss

The critic minimizes:

$$L_{critic} = \mathbb{E}[C(x_{fake})] - \mathbb{E}[C(x_{real})] + \lambda \cdot GP$$

The generator minimizes:

$$L_{generator} = -\mathbb{E}[C(x_{fake})]$$

The critic wants real scores high and fake scores low (minimizing the expression makes $C(x_{real})$ large and $C(x_{fake})$ small). The generator wants fakes to score high (minimizing $-\mathbb{E}[C(x_{fake})]$ pushes fake scores up).

No logarithms, no sigmoid. Just mean scores.

# 3. WGAN-GP Implementation

## The Critic

The Critic has the same architecture as the Discriminator, but uses **InstanceNorm instead of BatchNorm** to avoid batch-level dependencies. InstanceNorm normalizes each image independently, which is compatible with the gradient penalty computation.

Although both networks lack a final activation function, their outputs have different interpretations: the Discriminator's output is a logit (representing log-odds of being real, used with BCEWithLogitsLoss), while the Critic's output is an unbounded score (representing "realness" on a continuous scale).

**Note:** Following DCGAN conventions, we do **not** apply normalization to the first convolutional layer.

In [ ]:
class Critic(nn.Module):
    def __init__(self, d_dim=64, img_size=64):
        super().__init__()

        kernel_size = 4
        n = 4
        fc_in = int(d_dim * 2**(n-1) * (img_size/(2**n))**2)
        pad = 1
        stride = 2
        bias = False

        def conv(in_channels, out_channels):
            return nn.Conv2d(in_channels=in_channels,
                             out_channels=out_channels,
                             kernel_size=kernel_size,
                             stride=stride,
                             padding=pad,
                             bias=bias)

        self.fc = nn.Linear(fc_in, 1)

        self.conv1 = conv(in_channels=3, out_channels=d_dim)
        self.conv2 = conv(in_channels=d_dim, out_channels=2*d_dim)
        self.conv3 = conv(in_channels=2*d_dim, out_channels=4*d_dim)
        self.conv4 = conv(in_channels=4*d_dim, out_channels=8*d_dim)

        # InstanceNorm — no normalization on first layer
        self.inorm2 = nn.InstanceNorm2d(2 * d_dim)
        self.inorm3 = nn.InstanceNorm2d(4 * d_dim)
        self.inorm4 = nn.InstanceNorm2d(8 * d_dim)

        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, x):
        batch_size = x.size(0)

        # No normalization on first layer
        out = self.leaky_relu(self.conv1(x))                  # b x 64 x 32 x 32
        out = self.leaky_relu(self.inorm2(self.conv2(out)))   # b x 128 x 16 x 16
        out = self.leaky_relu(self.inorm3(self.conv3(out)))   # b x 256 x 8 x 8
        out = self.leaky_relu(self.inorm4(self.conv4(out)))   # b x 512 x 4 x 4

        out = out.contiguous().view(batch_size, -1)  # b x 8192
        scores = self.fc(out)  # b x 1
        return scores

## Wasserstein Loss

As explained in Section 2, the Wasserstein loss is simply the mean of the critic's scores, with sign depending on whether we're evaluating real or fake images.

In [ ]:
# Also available as hp.Wasserstein_loss_fcn in helper_L3.py.
def Wasserstein_loss_fcn(input_tensor, **kwargs):
    label_type = kwargs.get("label_type", "real").lower()

    if label_type not in ("real", "fake"):
        raise ValueError(
            f"label_type must be 'real' or 'fake', got '{label_type}'")

    sign = -1 if label_type == 'real' else 1
    return input_tensor.mean() * sign

## Gradient Penalty

The gradient penalty enforces the 1-Lipschitz constraint. It works by:
1. Creating **interpolated images** between real and fake (using random mixing coefficients).
2. Computing the **critic's gradient** with respect to these interpolated images.
3. Penalizing when the gradient norm deviates from 1.

Reference: [WGAN-GP implementation on GitHub](https://github.com/eriklindernoren/PyTorch-GAN/blob/master/implementations/wgan_gp/wgan_gp.py)

In [ ]:
# Also available as hp.penalty_fcn in helper_L3.py.
def penalty_fcn(real_img, fake_img, critic, gamma=10):
    """Calculate gradient penalty."""
    alpha = torch.rand(real_img.shape[0], 1, 1, 1, device=DEVICE)

    mix_images = torch.lerp(fake_img, real_img, alpha)
    mix_images.requires_grad_(True)

    mix_pred = critic(mix_images)

    gradients = torch.autograd.grad(
        inputs=mix_images,
        outputs=mix_pred,
        grad_outputs=torch.ones_like(mix_pred),
        retain_graph=True,
        create_graph=True,
        only_inputs=True)[0]

    gradients = gradients.view(len(gradients), -1)
    gradient_norm = torch.linalg.vector_norm(gradients, ord=2, dim=1)
    gp = ((gradient_norm - 1) ** 2).mean()

    return gp * gamma

## Generator

We use the same Generator architecture as in LESSON 3A. It is available from `helper_L3.py`.

In [ ]:
# Same Generator as in LESSON 3A — loaded from helper
Generator = hp.Generator

In [ ]:
# Instantiate models and optimizers
torch.manual_seed(SEED)
G = Generator()
C = Critic()

lr_g = 0.0002
lr_c = 0.0002
beta1 = 0.65
beta2 = 0.999

g_optimizer = torch.optim.Adam(G.parameters(), lr_g, [beta1, beta2])
c_optimizer = torch.optim.Adam(C.parameters(), lr_c, [beta1, beta2])

In [ ]:
# Training configuration — 40 epochs with WGAN-GP
# Note: hp.train() uses 'discriminator' and 'd_optimizer' keys for both classical
# GAN and WGAN-GP. For WGAN-GP, these refer to the Critic.
n_epochs = 40

config = {'n_epochs': n_epochs,
          'dataloader': dataloader,
          'discriminator': C,
          'generator': G,
          'd_optimizer': c_optimizer,
          'g_optimizer': g_optimizer,
          'project_dir': PROJECT_DIR,
          'loss_fcn': Wasserstein_loss_fcn,
          'show_step': 10,
          'penalty_fcn': penalty_fcn,
          'crit_cycles': 2,
          'save_step': 40,
          'save_starting': 1000,   # Effectively disabled — only save at save_step intervals
          'checkpoint_prefix': 'wgangp',
          }

In [ ]:
_ = hp.train(config)

# 4. Comparing Classical GAN and WGAN-GP

Let's load both checkpoints and generate images with the same noise to compare the two approaches.

To load the classical GAN checkpoint from LESSON 3A, we need the `Discriminator`
class definition. This is the same architecture used in LESSON 3A.

In [ ]:
# Discriminator from LESSON 3A (needed to load the classical GAN checkpoint)
class Discriminator(nn.Module):
    def __init__(self, d_dim=64, img_size=64):
        super().__init__()
        kernel_size = 4
        n = 4
        fc_in = int(d_dim * 2**(n-1) * (img_size / (2**n))**2)
        pad, stride, bias = 1, 2, False

        def conv(in_channels, out_channels):
            return nn.Conv2d(in_channels, out_channels,
                             kernel_size, stride, pad, bias=bias)

        self.fc = nn.Linear(fc_in, 1)
        self.conv1 = conv(3, d_dim)
        self.conv2 = conv(d_dim, d_dim * 2)
        self.conv3 = conv(d_dim * 2, d_dim * 4)
        self.conv4 = conv(d_dim * 4, d_dim * 8)
        self.bnorm2 = nn.BatchNorm2d(d_dim * 2)
        self.bnorm3 = nn.BatchNorm2d(d_dim * 4)
        self.bnorm4 = nn.BatchNorm2d(d_dim * 8)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, x):
        batch_size = x.size(0)
        out = self.leaky_relu(self.conv1(x))
        out = self.leaky_relu(self.bnorm2(self.conv2(out)))
        out = self.leaky_relu(self.bnorm3(self.conv3(out)))
        out = self.leaky_relu(self.bnorm4(self.conv4(out)))
        out = out.contiguous().view(batch_size, -1)
        return self.fc(out)

In [ ]:
# Load classical GAN checkpoint (trained in LESSON 3A)
# Note: You need to have run LESSON 3A first to have this checkpoint.
# The Discriminator class is defined above for this purpose.
# The classical GAN was saved at epoch 40.

# Instantiate fresh models for classical GAN loading
D_classical = Discriminator()
G_classical = Generator()
d_opt_classical = torch.optim.Adam(D_classical.parameters(), 0.0004, [0.6, 0.999])
g_opt_classical = torch.optim.Adam(G_classical.parameters(), 0.0004, [0.6, 0.999])

config_classical = {
    'discriminator': D_classical,
    'generator': G_classical,
    'd_optimizer': d_opt_classical,
    'g_optimizer': g_opt_classical,
}

# Try loading the classical GAN checkpoint
result = hp.load_checkpoint('classical_epoch_40', config_classical, PROJECT_DIR)

if result is not None:
    # Generate with same fixed noise for fair comparison
    z_dim = 100  # Must match Generator's default z_dim
    fixed_noise = torch.randn(25, z_dim, device=DEVICE)

    G_classical.to(DEVICE).eval()
    G.eval()

    with torch.no_grad():
        imgs_classical = G_classical(fixed_noise)
        imgs_wgangp = G(fixed_noise)

    # Side-by-side comparison
    hp.visual_comparison(imgs_classical, imgs_wgangp,
                         title_a="Classical GAN (epoch 40)",
                         title_b="WGAN-GP (epoch 40)")
else:
    print("Classical GAN checkpoint not found. Run LESSON 3A first.")

## Comparing Loss Curves

The WGAN-GP loss curves are typically much smoother and more stable than classical GAN loss curves. The Wasserstein loss provides a meaningful metric of training progress, unlike the BCE loss which can oscillate wildly.

<img src="images/loss_curves_comparison.png" width=1200, align="center"/>

**Key observations:**
- **Stability**: WGAN-GP training is smoother — the loss curves show a clear convergence trend rather than the oscillation typical of classical GANs.
- **Quality**: WGAN-GP often produces sharper, more detailed images.
- **Convergence**: The Wasserstein critic loss provides a meaningful metric of training progress. As the generator improves, both losses converge: the generator loss decreases toward zero while the critic loss stabilizes. Unlike BCE loss, the Wasserstein loss gives a reliable signal throughout training.

## Conclusion

In this notebook we learned:
1. **Why classical GANs fail**: BCE loss leads to vanishing gradients when the discriminator is too good.
2. **Wasserstein distance**: Provides smooth gradients even for non-overlapping distributions.
3. **WGAN-GP**: Uses gradient penalty to enforce the Lipschitz constraint, enabling stable training.

In **LAB 3**, you'll apply these concepts hands-on: building larger architectures, comparing training approaches, exploring the latent space, investigating hyperparameter sensitivity, and analyzing mode collapse.